# le fat chaton — Modal notebook (smol-fat, auto-save, auto-release GPU)

Trains the smol-fat MoE on an **L4 GPU** and **auto-releases the GPU when done**
(so billing stops the instant training finishes — walk away, it won't keep
charging). Runs the heavy work via a `.remote()` Modal function — the notebook's
own session is just the cheap orchestrator.

**Auto-save**: checkpoints push to HuggingFace Hub `mateo0093/le-fat-chaton-ckpt`
every 250 steps + final `model.pt` to the persistent Modal volume `/ckpt`.

## Setup (do ONCE before running)
1. Create a Modal secret named **`chaton-hf`** (modal.com → Secrets → New)
   with key `HF_TOKEN` = your HuggingFace write token.
2. Attach the `chaton-hf` secret to THIS notebook (top toolbar → Secrets → add).
3. Select an **L4 GPU** for the notebook session (or leave CPU — the L4 is
   provisioned inside the `.remote()` call).

**Run cells top to bottom.** Cell 3 is the long one — it streams live training
steps and finishes when the L4 container exits (= GPU released).

In [ ]:
# 1. Install the Modal client in the notebook session (if not already there)
%pip install -q modal

In [ ]:
# 2. Confirm the secret is attached. If this prints False, attach chaton-hf first.
import os
print("HF_TOKEN present:", "HF_TOKEN" in os.environ,
      "(must be True -> chaton-hf secret is attached)")

In [ ]:
# 3. DEFINE + LAUNCH the L4 training run. This streams steps live and
#    AUTO-RELEASES the L4 the moment train.remote() returns.
import os, modal, subprocess, sys

app = modal.App("le-fat-chaton-train")
ckpt_vol = modal.Volume.from_name("le-fat-chaton-ckpt", create_if_missing=True)

image = (
    modal.Image.debian_slim(python_version="3.11")
    .apt_install("git", "build-essential")
    .pip_install("torch", "tiktoken", "datasets==2.21.0", "huggingface_hub<0.30",
                 "tokenizers", "numpy")
)


@app.function(
    image=image,
    gpu="L4",                        # 24GB, ~$0.80/hr, cost-efficient
    timeout=60 * 60,                 # 1h safety cap (run ~15-25 min on L4)
    volumes={"/ckpt": ckpt_vol},     # persistent -> survives container exit
    secrets=[modal.Secret.from_name("chaton-hf")],   # injects HF_TOKEN
)
def train():
    import os, subprocess, sys
    os.chdir("/root")
    subprocess.run(["git", "clone", "https://github.com/Mateooo93/le-gros-chaton.git"],
                   check=True)
    os.chdir("/root/le-gros-chaton")

    env = dict(os.environ)
    env.update({
        "CHATON_PROFILE":       "smol-fat",     # 240M / 63M-active MoE
        "CHATON_BLOCK_SIZE":    "1024",         # fits L4 24GB comfortably
        "CHATON_MICRO_BATCH":   "8",
        "CHATON_GRAD_ACCUM":    "8",            # eff batch = 64
        "CHATON_COMPILE":       "1",            # ON (L4 has enough SMs)
        "CHATON_MAX_ITERS":     "1000",
        "CHATON_CKPT_INTERVAL": "250",          # push to Hub every 250 steps
        "CHATON_RESUME":        "1",            # pull latest, resume if present
        "CHATON_DATA":          "wikitext",     # durability proof on wikitext
        "CHATON_CKPT_PATH":     "/ckpt/checkpoint.pt",
        "CHATON_HF_REPO":       "mateo0093/le-fat-chaton-ckpt",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    })
    print("[modal] HF_TOKEN present:", "HF_TOKEN" in env, "| starting train.py ...")

    p = subprocess.Popen([sys.executable, "-u", "train.py"], env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    print("[modal] train.py exit:", rc)
    ckpt_vol.commit()                  # persist final state to the volume
    print("[modal] volume committed -> /ckpt survives container exit")
    return rc


with app.run():
    rc = train.remote()
    print(f"\n[notebook] DONE, exit={rc}. L4 container released -> GPU billing stopped.")

## What "auto-save + delete kernel when done" means here
- **Auto-save**: every 250 steps the checkpoint (model + optimizer + step +
  scaler) goes to HF Hub `mateo0093/le-fat-chaton-ckpt`, and the final
  `model.pt` is committed to the persistent Modal volume `/ckpt`.
- **Delete kernel when done**: the L4 GPU container is created by
  `train.remote()` and is **automatically destroyed by Modal when the function
  returns**. GPU billing stops that second. You don't have to stop anything.

## To resume after a disconnect
Just re-run cell 3. `CHATON_RESUME=1` pulls the latest checkpoint from the Hub
and continues from that step. No progress lost.